### Bill summarization using Pegasus Model on BillSum Dataset
----
  #### Task List:
  - Load the dataset
  - Load the tokenizer and model
  - Define tokenization function
  - Apply tokenization on train, test and validation
  - Define rough metric function
  - Define training arguments
  - Define data-collator
  - Define trainer and start training
  - Save the tokenizer and model
  - Do the test on test dataset
  - Make a function to do inference
  - Do inference
--- 

#### Load the dataset

In [1]:
from datasets import load_dataset
dataset = load_dataset("FiscalNote/billsum")

In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 1237
    })
})

In [3]:
print(dataset['train'][0]['text'])
print(" = " * 20)
print(dataset['train'][0]['summary'])

SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES 
              TO NONPROFIT ORGANIZATIONS.

    (a) Definitions.--In this section:
            (1) Business entity.--The term ``business entity'' means a 
        firm, corporation, association, partnership, consortium, joint 
        venture, or other form of enterprise.
            (2) Facility.--The term ``facility'' means any real 
        property, including any building, improvement, or appurtenance.
            (3) Gross negligence.--The term ``gross negligence'' means 
        voluntary and conscious conduct by a person with knowledge (at 
        the time of the conduct) that the conduct is likely to be 
        harmful to the health or well-being of another person.
            (4) Intentional misconduct.--The term ``intentional 
        misconduct'' means conduct by a person with knowledge (at the 
        time of the conduct) that the conduct is harmful to the health 
        or well-being of another perso

#### Load the tokenizer and model

In [4]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [5]:
from transformers import PegasusTokenizerFast, PegasusForConditionalGeneration

In [6]:
model_checkpoint = "google/pegasus-xsum"
tokenizer = PegasusTokenizerFast.from_pretrained(model_checkpoint)
model = PegasusForConditionalGeneration.from_pretrained(model_checkpoint).to(device)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Define tokenization function

In [7]:
TEXT_MAX_LEN = 512
SUMMARY_MAX_LEN = 128

def tokenization(batch):
    # get the text and summary
    texts = batch['text']
    summaries = batch['summary']
    
    # tokenize text and summary
    tokenized_texts = tokenizer(
        texts,
        padding = 'max_length',
        max_length = TEXT_MAX_LEN,
        truncation = True
    )
    tokenized_summaries = tokenizer(
        summaries,
        padding = 'max_length',
        max_length = SUMMARY_MAX_LEN,
        truncation = True
    )
    
    # replace pad index with -100
    summaries_ids = tokenized_summaries['input_ids']
    summaries_ids = [
        [(token if token != tokenizer.pad_token_id else -100)
          for token in tokens 
        ]
        for tokens in summaries_ids
    ]
    
    # format the dict and return
    tokenized_texts['labels'] = summaries_ids
    return tokenized_texts

In [8]:
tokenized_dataset = dataset.map(
    tokenization , batched = True
)

Map:   0%|          | 0/3269 [00:00<?, ? examples/s]

In [9]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1237
    })
})

### Define evaluation metric

In [10]:
import evaluate
import numpy as np
rouge_metric = evaluate.load("rouge")

In [11]:
def compute_metrics(eval_preds):
    # get the predictions and labels
    preds , labels = eval_preds
    
    # replace -100 with pad_token_id
    preds = np.where(preds != -100 , preds , tokenizer.pad_token_id)
    labels = np.where(labels != -100 , labels , tokenizer.pad_token_id)
    
    # decode both predictions and labels
    preds_decoded = tokenizer.batch_decode(preds , skip_special_tokens = True)
    labels_decoded = tokenizer.batch_decode(labels , skip_special_tokens = True)
    
    # compute result
    result = rouge_metric.compute(
        predictions = preds_decoded,
        references = labels_decoded,
        use_stemmer = True,
        use_aggregator = True
    )
    
    # convert into percentage and round
    result = {
        key: round(val * 100 , 2) for key , val in result.items()
    }
    return result

### Define training arguments

In [12]:
from transformers import Seq2SeqTrainingArguments

In [13]:
train_args = Seq2SeqTrainingArguments(
    output_dir = './pegasus-result',
    
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    num_train_epochs = 2,
    
    eval_strategy = 'steps',
    eval_steps = 500,
    
    logging_strategy = 'steps',
    logging_steps = 500,
    
    save_strategy = 'steps',
    save_steps = 500,
    
    learning_rate = 2e-5,
    weight_decay = 0.001,
    
    predict_with_generate = True,
    generation_max_length = 128,
    generation_num_beams = 4,
    
    load_best_model_at_end = True,
    metric_for_best_model = 'rougeL',
    greater_is_better = True
)

### Define DataCollator

In [14]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer = tokenizer,
    model = model
)

### Define trainer

In [15]:
from transformers import Seq2SeqTrainer

In [16]:
trainer = Seq2SeqTrainer( 
    model = model,
    args = train_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test'],
    compute_metrics = compute_metrics,
    data_collator = data_collator
)

In [ ]:
trainer.train()